This code aims **`to identify credit card customers who might be considered 'skippers' or 'defaulters'`** by assigning points based on their monthly payment behavior. The more points a customer accumulates, the more likely they are to be identified as a defaulter.

Here's how the points are assigned:

**Short Payment Penalty:** A customer receives 1 point if they fail to pay at least 70% of their total monthly spending. This rule targets customers who significantly underpay their bills.

**Max Limit Exceeded and Not Fully Cleared:** A customer also receives 1 point if they spend 100% of their credit limit but then do not pay the full outstanding amount for that month. This rule focuses on customers who utilize their full credit limit but then struggle to clear the balance.

**Double Trouble Bonus:** If a customer meets both of the above conditions in the same month (i.e., they make a short payment AND exceeded their max limit without fully clearing the amount), they are assigned an additional 1 point, bringing their total for that month's specific behavior to 3 points (1 for short payment + 1 for max limit + 1 additional).

Finally, *for each customer, the code will sum up all the points they have accumulate*d across different months based on these rules. This total score is then written to an output file, allowing for further analysis or action based on the customers with higher scores.

In [ ]:
!pip install --quiet apache-beam[gcp]

In [ ]:
from google.colab import files
files.upload()

In [4]:
import apache_beam as beam
import logging
p1=beam.Pipeline()

class AssignPoints(beam.DoFn):
  def process(self,element):
      element_list=element.split(',')
      try:
          # Ensure element_list has enough indices before accessing
          if len(element_list) < 9: # Need up to index 8 for cleared_amount
              logging.warning(f"Skipping malformed element: Not enough columns. Element: {element}")
              return [] # Return empty list to skip this element

          max_credit_limit=int(element_list[5])
          total_spent=int(element_list[6])
          cleared_amount=int(element_list[8])
      except (ValueError, IndexError) as e:
          logging.error(f"Error processing element during data conversion or access: {element}. Details: {e}")
          return [] # Return empty list to skip this malformed element

      assigned_point=0
      is_short_payment = False
      is_max_limit_exceeded_and_not_cleared = False

      # Short Payment Penalty: 1 point if cleared_amount <= 70% of total_spent
      if cleared_amount <= int(total_spent*0.7):
          assigned_point+=1
          is_short_payment = True

      # Max Limit Exceeded and Not Fully Cleared: 1 point if max_credit_limit == total_spent and cleared_amount < total_spent
      if (max_credit_limit == total_spent and cleared_amount < total_spent):
            assigned_point+=1
            is_max_limit_exceeded_and_not_cleared = True

      # Double Trouble Bonus: Additional 1 point if both above conditions are met
      if is_short_payment and is_max_limit_exceeded_and_not_cleared:
          assigned_point+=1 # Additional point as per problem description

      # Return a list containing an explicit (key, value) tuple for CombinePerKey
      return [(element_list[0]+" "+element_list[1], assigned_point)]


cards_pcoll=(p1
             |'Read inpu pcoll'>>beam.io.ReadFromText('/content/cards.txt',skip_header_lines=1)
             |'Assign points'>>beam.ParDo(AssignPoints())
             |'Combine points per customer'>>beam.CombinePerKey(sum)
             |'Output to file'>>beam.io.WriteToText('/content/assigned_points.txt')

)
p1.run()
!{('head -n 10 /content/assigned*')}

('CT28383 Miyako', 3)
('CT74474 Nanaho', 3)
('CT66322 Chris', 1)
('CT65528 Bonnie', 2)
('CT84463 Isaac', 4)
('CT12838 Isidore', 5)
('CT96185 Danielle', 3)
('CT74827 Hanna', 3)
('CT98239 Sayuri', 4)
('CT57141 Kaori', 2)
